In [ ]:
%load_ext autoreload
%autoreload 2

### 1. 모듈 임포트

In [ ]:
import torch
import numpy as np
# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features


In [ ]:
# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. 데이터셋 생성

In [ ]:
import numpy as np
from modules.sdf_generator import create_random_shape, to_grid
from modules.particle_sampler import sample_particles_poisson  # 여기를 변경!
from modules.visualizer import visualize_simulation

# --- 설정 ---
NUM_SAMPLES = 5         # 테스트할 개수
RESOLUTION = 64         # 그리드 해상도
NUM_PARTICLES = 2000    # 목표 파티클 개수 (도형 내부를 채울 개수)

print(f"🚀 Poisson Disk Sampling 테스트 시작 ({NUM_SAMPLES}개)")

for i in range(NUM_SAMPLES):
    seed = 2024 + i  # 시드값 변경 (매번 다른 모양)
    
    # 1. 랜덤 도형 생성
    print(f"\n[{i+1}/{NUM_SAMPLES}] 도형 생성 중 (Seed: {seed})...")
    random_shape = create_random_shape(seed=seed)
    sdf_grid = to_grid(random_shape, resolution=RESOLUTION, domain_size=2.0)
    
    # 2. Poisson Disk Sampling 실행
    # (아까 수정한 로직 덕분에, 내부 부피를 계산해서 약 2000개를 맞춰줍니다)
    particles = sample_particles_poisson(
        sdf_grid, 
        domain_size=2.0, 
        num_particles=NUM_PARTICLES
    )
    
    print(f"   -> 생성된 파티클: {len(particles)}개")
    
    # 3. 시각화 (왼쪽: 단면, 오른쪽: 3D)
    visualize_simulation(
        sdf_grid=sdf_grid, 
        particles=particles, 
        domain_size=2.0, 
        title=f"Poisson Sample {i+1} (N={len(particles)})"
    )

print("\n✅ 테스트 완료!")

### 2.1 데이터 셋 파티클 Poisson Disk 샘플링

In [ ]:

# my shape -> sdf 해상도 64로 설정
sdf_grid = to_grid(my_shape, resolution=RESOLUTION, domain_size=2.0)

# 입자 샘플링을 위해 Numpy 포맷으로 저장
np.save("sdf_grid_64.npy", sdf_grid)

print(f"Grid Shape: {sdf_grid.shape}")
print(f"Min Value: {sdf_grid.min():.3f}, Max Value: {sdf_grid.max():.3f}")

In [ ]:
import os
import numpy as np
from modules.sdf_generator import create_random_shape, to_grid
from modules.visualizer import visualize_simulation # 시각화용

# --- 설정값 ---
DATASET_SIZE = 5      # 생성할 데이터 개수
START_SEED = 42       # 시작 시드값 (재현 가능)
OUTPUT_DIR = "dataset" # 저장할 폴더

# 폴더 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 랜덤 데이터셋 생성 시작 (수량: {DATASET_SIZE}개)")

for i in range(DATASET_SIZE):
    # 1. 시드 설정 (각 데이터마다 시드가 달라야 함)
    current_seed = START_SEED + i
    
    # 2. 랜덤 도형 생성
    print(f"[{i+1}/{DATASET_SIZE}] Generating Shape (Seed: {current_seed})...")
    random_sdf = create_random_shape(seed=current_seed)
    
    # 3. 그리드로 변환
    sdf_grid = to_grid(random_sdf, resolution=RESOLUTION, domain_size=2.0)
    
    # 4. 저장 (NPY)
    # 파일명: shape_000.npy, shape_001.npy ...
    filename = os.path.join(OUTPUT_DIR, f"shape_{i:03d}.npy")
    np.save(filename, sdf_grid)
    
    print(f"\n👀 {i} 번째 데이터 미리보기:")
    visualize_simulation(sdf_grid=sdf_grid, title=f"Sample {i} (Seed {current_seed})")

print(f"\n✅ 생성 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

## 3. sdf 계산 -CNN 학습

### 3.1 전처리 단계: 그리드 특징값(m_c) 계산

In [ ]:
# 1. 모듈 임포트
import torch
import numpy as np
from modules.sdf_network import (
    FeatureConstruction, 
    SDFNetwork, 
    SDFReconstruction
)
from modules.visualizer import (
    visualize_feature_grid,
    visualize_sdf,
    visualize_particles_and_features,
    visualize_simulation
)
from modules.sdf_generator import sphere, to_grid
from modules.particle_sampler import sample_particles_poisson
import matplotlib.pyplot as plt

# 2. 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

# 3. SDF 생성 및 파티클 샘플링 (테스트 케이스)
print("\n=== [Step 1-0] 테스트 케이스: SDF 구(Sphere) 생성 ===")

# 3-1. 구 SDF 생성
sphere_sdf = sphere(0.8)  # 반지름 0.8의 구
sdf_grid_sphere = to_grid(sphere_sdf, resolution=RESOLUTION, domain_size=2.0)
print(f"✓ 구 SDF 생성 완료: {sdf_grid_sphere.shape}")
print(f"  - SDF 범위: [{sdf_grid_sphere.min():.3f}, {sdf_grid_sphere.max():.3f}]")

# 3-2. Poisson Disk Sampling으로 구 내부 파티클 생성
print("\n=== [Step 1-1] Poisson Disk Sampling으로 파티클 생성 ===")
particles_np = sample_particles_poisson(
    sdf_grid_sphere,
    domain_size=2.0,
    num_particles=4000  # 약 4000개 파티클
)
print(f"✓ 파티클 생성 완료: {len(particles_np)}개")

# 3-3. 파티클을 torch tensor로 변환
particles = torch.tensor(
    particles_np,
    dtype=torch.float32,
    device=device
)

print(f"✓ 파티클 데이터 변환 완료: {particles.shape}")
print(f"  - 좌표 범위: X=[{particles[:, 0].min():.3f}, {particles[:, 0].max():.3f}]")
print(f"                 Y=[{particles[:, 1].min():.3f}, {particles[:, 1].max():.3f}]")
print(f"                 Z=[{particles[:, 2].min():.3f}, {particles[:, 2].max():.3f}]")

# 3-4. 생성 결과 시각화 (SDF와 파티클 함께)
print("\n=== [Step 1-2] 생성된 구와 파티클 시각화 ===")
visualize_simulation(
    sdf_grid=sdf_grid_sphere,
    particles=particles_np,
    domain_size=2.0,
    title="테스트 케이스: 구(Sphere) + Poisson Sampled Particles"
)

In [ ]:
print("\n=== [Step 2] 전처리: 그리드 특징값(m_c) 계산 ===")

# 4. 전처리 초기화 (그리드 간격 설정)
dx = 0.2  # 그리드 간격
feature_construction = FeatureConstruction(dx=dx, device=device)
print(f"✓ FeatureConstruction 설정 완료")
print(f"  - 그리드 간격 (dx): {dx}")
print(f"  - 커널 반경 (R): {feature_construction.R:.3f}")

# 5. 그리드 생성 및 특징값 계산
print("\n  실행 중...")
grid_nodes, m_c, grid_shape = feature_construction(particles)

print(f"✓ 전처리 완료")
print(f"  - 그리드 크기: {grid_shape}")
print(f"  - 그리드 노드: {grid_nodes.shape}")
print(f"  - m_c 통계")
print(f"    * Min: {m_c.min():.6f}")
print(f"    * Max: {m_c.max():.6f}")
print(f"    * Mean: {m_c.mean():.6f}")
print(f"    * Std: {m_c.std():.6f}")

### 3.2 m_c 값 시각화

In [ ]:
print("\n=== [Step 3] 시각화 1: 그리드 특징값(m_c) ===\n")

# 파티클과 그리드 특징값 동시 시각화
fig1 = visualize_particles_and_features(
    particles=particles,
    grid_nodes=grid_nodes,
    m_c_grid=m_c.reshape(grid_shape),
    title="particle(red) and grid feature value(colormapped)"
)

# 그리드 특징값만 시각화
fig2 = visualize_feature_grid(
    m_c_grid=m_c.reshape(grid_shape),
    grid_nodes=grid_nodes,
    title="grid feature value m_c (Poly6 kernel based)"
)

print("✓ m_c 시각화 완료")

### 3.3 3D CNN 네트워크를 통한 SDF 값 추론

In [ ]:
print("\n=== [Step 4] 3D CNN 네트워크 구성 및 SDF 추론 ===\n")

# 6. 통합 파이프라인 초기화
reconstruction = SDFReconstruction(dx=dx, device=device)

print(f"✓ SDFReconstruction 초기화 완료")
print(f"  - 네트워크 구조:")
print(f"    * Input: (Batch, 1, 8, 8, 8)")
print(f"    * Conv3d(1→32) + BN + LeakyReLU")
print(f"    * Conv3d(32→64) + BN + LeakyReLU")
print(f"    * MaxPool3d(2×2×2)")
print(f"    * Conv3d(64→128) + BN + LeakyReLU")
print(f"    * Conv3d(128→256) + BN + LeakyReLU")
print(f"    * Flatten")
print(f"    * FC(16384→256) + LeakyReLU + Dropout")
print(f"    * FC(256→128) + LeakyReLU + Dropout")
print(f"    * FC(128→1) → SDF 값\n")

# 7. SDF 추론 (배치 처리)
print("  SDF 값 계산 중...")
grid_nodes_full, sdf_values, _, _ = reconstruction.forward(
    particle_positions=particles,
    num_inference_nodes=500,  # 처음 500개 노드만 추론 (데모용)
    batch_size=32
)

print(f"✓ SDF 추론 완료")
print(f"  - SDF 값 통계")
print(f"    * Min: {sdf_values.min():.6f}")
print(f"    * Max: {sdf_values.max():.6f}")
print(f"    * Mean: {sdf_values.mean():.6f}")
print(f"    * Std: {sdf_values.std():.6f}")

### 3.4 학습

In [ ]:
# SDFNetwork 학습 예제
import torch
from torch.utils.data import DataLoader, TensorDataset
from modules.sdf_network import SDFNetwork

# 예시 데이터 준비 (실제 프로젝트에서는 m_c_patch, sdf_gt를 생성해야 함)
# 아래는 데모용 랜덤 데이터
batch_size = 16
num_samples = 128
patch_shape = (1, 8, 8, 8)

# 랜덤 feature patch와 SDF GT 생성
feature_patches = torch.randn(num_samples, *patch_shape)
sdf_gt = torch.randn(num_samples)

dataset = TensorDataset(feature_patches, sdf_gt)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 네트워크, 옵티마이저, 손실함수 정의
model = SDFNetwork().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# 학습 실행
num_epochs = 5
losses = model.train_step(train_loader, optimizer, criterion, device=device, num_epochs=num_epochs)

print("학습 완료. 에폭별 손실:", losses)
# 학습된 모델로 추론 예시
# (여기서는 랜덤 feature patch 중 일부로 추론)
model.eval()
with torch.no_grad():
    test_patch = feature_patches[0:1].to(device)  # 첫 번째 패치
    pred_sdf = model(test_patch)
    print(f"추론 결과 SDF 값: {pred_sdf.item():.6f}")

## 4. SDF 값 시각화 (최종 결과)

In [ ]:
print("\n=== [Step 5] 시각화 2: SDF 재구성 결과 ===\n")

# 8. SDF 값 시각화
fig3 = visualize_sdf(
    sdf_values=sdf_values[:grid_nodes_full.shape[0] if sdf_values.shape[0] < grid_nodes_full.shape[0] else sdf_values.shape[0]],
    grid_nodes=grid_nodes_full[:sdf_values.shape[0]],
    title="3D CNN 디코더 출력 SDF 값"
)
plt.show()

print("✓ SDF 재구성 완료\n")

# 9. 요약
print("=" * 60)
print("✅ 3D SDF 재구성 파이프라인 완료!")
print("=" * 60)
print(f"\n📊 처리 결과 요약:")
print(f"  [입력]")
print(f"    - 파티클 수: {particles.shape[0]}")
print(f"    - 파티클 좌표 범위: [{particles.min():.3f}, {particles.max():.3f}]")
print(f"\n  [전처리 (Poly6 커널)]")
print(f"    - 그리드 간격 (dx): {dx}")
print(f"    - 커널 반경 (R): {feature_construction.R:.3f}")
print(f"    - 그리드 노드 수: {grid_nodes_full.shape[0]}")
print(f"    - m_c 범위: [{m_c.min():.6f}, {m_c.max():.6f}]")
print(f"\n  [네트워크 (3D CNN)]")
print(f"    - 입력 해상도: 8×8×8")
print(f"    - 추론된 노드: {sdf_values.shape[0]}")
print(f"    - SDF 범위: [{sdf_values.min():.6f}, {sdf_values.max():.6f}]")
print(f"\n  [출력]")
print(f"    - 형태: 레벨셋 (Level Set)")
print(f"    - 표현: 각 그리드 노드에서 SDF 값")

In [ ]:
print("\n=== [Bonus] 실제 파티클 샘플러로부터의 데이터 파이프라인 ===\n")

from modules.sdf_generator import create_random_shape, to_grid
from modules.particle_sampler import sample_particles_poisson

# 1. 랜덤 도형 생성
print("[1] 랜덤 도형 생성...")
random_sdf = create_random_shape(seed=2024)
sdf_grid_test = to_grid(random_sdf, resolution=RESOLUTION, domain_size=2.0)

# 2. Poisson Disk Sampling으로 파티클 생성
print("[2] Poisson Disk Sampling으로 파티클 생성...")
particles_real = sample_particles_poisson(
    sdf_grid_test,
    domain_size=2.0,
    num_particles=2000
)
print(f"    ✓ 생성된 파티클: {len(particles_real)}개")

# 3. 파티클 to 텐서 변환
particles_tensor = torch.tensor(
    particles_real,
    dtype=torch.float32,
    device=device
)

# 4. SDF 재구성 파이프라인 실행
print("[3] SDF 재구성 파이프라인 실행...")
reconstruction_test = SDFReconstruction(dx=0.15, device=device)

grid_nodes_test, sdf_values_test, _, _ = reconstruction_test.forward(
    particle_positions=particles_tensor,
    num_inference_nodes=300,
    batch_size=32
)

print(f"    ✓ 그리드 노드: {grid_nodes_test.shape}")
print(f"    ✓ SDF 값: {sdf_values_test.shape}")
print(f"    ✓ SDF 범위: [{sdf_values_test.min():.6f}, {sdf_values_test.max():.6f}]")

# 5. 시각화
print("[4] 결과 시각화...")
fig_combo = visualize_particles_and_features(
    particles=particles_tensor,
    grid_nodes=grid_nodes_test,
    m_c_grid=reconstruction_test.feature_construction.compute_grid_features(
        particles_tensor, 
        grid_nodes_test
    ).reshape(int(np.cbrt(grid_nodes_test.shape[0])), 
              int(np.cbrt(grid_nodes_test.shape[0])), 
              int(np.cbrt(grid_nodes_test.shape[0]))),
    title="실제 샘플 데이터: 파티클 + 그리드 특징값"
)
plt.show()

print("✅ 파이프라인 테스트 완료!")

### 4.2 obj로 저장(mc 알고리즘)

In [ ]:
from modules.visualizer import save_mesh_as_obj # 시각화용
save_mesh_as_obj(sdf_grid, filename="test_shape_64.obj")


npy 파일 이용한 sdf 시각화

In [ ]:
from modules.visualizer import visualize_npy
# 2. 데이터 시각화 (Phase 2)
print("\n--- Phase 2: 시각화 및 저장 ---")

# SDF 파일 확인
visualize_npy("sdf_grid_64.npy")

# 파티클 파일 확인
visualize_npy("particles.npy")


변수(객체) 이용한 시각화(런타임에 있는 메모리)

In [ ]:
from modules.visualizer import visualize_simulation

# SDF만 보기 (기존 plot_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid)

# 파티클까지 겹쳐서 보기 (기존 plot_particles_and_sdf_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid, particles=particles)

obj 인터랙티브 뷰어

In [ ]:
from modules.visualizer import view_obj_interactive
# OBJ 파일 인터랙티브 뷰어
view_obj_interactive("test_shape_64.obj")

In [ ]:
import torch
import numpy as np

# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features

# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# Phase 1: SDF 도형 생성 및 래스터라이징
# ==========================================
print("\n[Phase 1] 🎲 랜덤 SDF 도형 생성 중...")
# 새로 만든 OOP 클래스를 사용합니다.
generator = SDFGenerator(config)

random_sdf = generator.create_random_shape(seed=2024) 
sdf_grid = generator.to_grid(random_sdf)

print(f"✅ SDF Grid 생성 완료: Shape={sdf_grid.shape}")

# ==========================================
# Phase 2: Poisson Disk 샘플링
# ==========================================
print("\n[Phase 2] 🎯 파티클 샘플링 중...")
# config 객체를 두 번째 인자로 넘겨받도록 변경된 부분을 반영합니다.
particles = sample_particles_poisson(sdf_grid, config)

print(f"✅ 파티클 샘플링 완료: {len(particles)}개 생성됨 (목표: {config.num_particles}개)")

# ==========================================
# Phase 3: 특징 인코딩 (Feature Construction)
# ==========================================
print("\n[Phase 3] 🧠 모델 입력용 특징(m_c) 추출 중...")
# 함수형 형태를 유지한 원본 FeatureConstruction을 사용하되, dx는 config 참조
feature_constructor = FeatureConstruction(dx=config.dx, device=device)

particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)

# 파티클 위치 기반으로 동적 그리드 변환 
grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
m_c_grid = m_c.reshape(grid_shape).cpu().numpy()

print(f"✅ 기하학적 특징 추출 완료")
print(f"   - Grid Shape: {grid_shape}")
print(f"   - m_c 통계: Min={m_c.min():.4f}, Max={m_c.max():.4f}")

# ==========================================
# Phase 4: 시각화 모듈
# ==========================================
print("\n[Phase 4] 📊 프로세스 검증용 데이터 시각화")

# 4-1. 전반적인 형상 + 파티클 위치 뷰어
visualize_simulation(
    sdf_grid=sdf_grid, 
    particles=particles, 
    domain_size=config.domain_size, 
    title="SDF Model & Generated Particles"
)

# 4-2. 파티클 분포와 네트워크의 m_c 추출 결과 뷰어
visualize_particles_and_features(
    particles=particles_tensor.cpu(),
    grid_nodes=grid_nodes.cpu(),
    m_c_grid=m_c_grid,
    title="Network Feature Distribution (m_c)"
)
